# Notebook 2 — Data Preprocessing
## Telecom Customer Retention Intelligence Platform

**Objective:** Clean the dataset, encode categorical variables,
handle class imbalance, and prepare train/test splits
ready for machine learning.

**Input:** data/raw/telco_churn.csv  
**Output:** data/processed/telco_churn_cleaned.csv  


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## 1. Load Raw Data

In [3]:
df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')

## 2. Fix Data Quality Issues

### 2.1 Fix TotalCharges — Convert to Numeric
TotalCharges contains blank strings for new customers.
We convert to numeric and fill missing values with the median.

In [4]:
# Convert blank strings to NaN then to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many NaN values were created
print(f"NaN values in TotalCharges after conversion: "
      f"{df['TotalCharges'].isnull().sum()}")

# Fill NaN with median — safer than mean for skewed distributions
median_total = df['TotalCharges'].median()
df['TotalCharges'].fillna(median_total, inplace=True)

print(f"Median used for imputation: ${median_total:.2f}")
print(f"NaN values after fix: {df['TotalCharges'].isnull().sum()}")

NaN values in TotalCharges after conversion: 11
Median used for imputation: $1397.47
NaN values after fix: 0


### 2.2 Fix SeniorCitizen — Standardize Encoding
SeniorCitizen is stored as 0/1 while all other binary 
columns use Yes/No. We standardize for consistency.

In [5]:
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

print("SeniorCitizen after fix:")
print(df['SeniorCitizen'].value_counts())

SeniorCitizen after fix:
SeniorCitizen
No     5901
Yes    1142
Name: count, dtype: int64


### 2.3 Drop Irrelevant Column
customerID is a unique identifier with no predictive value.
It must be removed before encoding and modeling.

In [6]:
df.drop(columns=['customerID'], inplace=True)
print(f"Shape after dropping customerID: {df.shape}")
print("customerID removed successfully")

Shape after dropping customerID: (7043, 20)
customerID removed successfully


## 3. Encode Categorical Variables

### 3.1 Label Encode Binary Columns
Binary columns (Yes/No, Male/Female) are converted to 0 and 1.

In [7]:
label_encode_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService',
    'PaperlessBilling', 'SeniorCitizen', 'Churn'
]

le = LabelEncoder()

for col in label_encode_cols:
    df[col] = le.fit_transform(df[col])
    print(f"{col}: {dict(zip(le.classes_, 
                             le.transform(le.classes_)))}")

gender: {'Female': np.int64(0), 'Male': np.int64(1)}
Partner: {'No': np.int64(0), 'Yes': np.int64(1)}
Dependents: {'No': np.int64(0), 'Yes': np.int64(1)}
PhoneService: {'No': np.int64(0), 'Yes': np.int64(1)}
PaperlessBilling: {'No': np.int64(0), 'Yes': np.int64(1)}
SeniorCitizen: {'No': np.int64(0), 'Yes': np.int64(1)}
Churn: {'No': np.int64(0), 'Yes': np.int64(1)}


### 3.2 One Hot Encode Multi-Category Columns
Columns with 3+ categories are one hot encoded to avoid
implying false numerical ordering between categories.

In [8]:
ohe_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod'
]

df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

print(f"Shape after one hot encoding: {df.shape}")
print(f"\nAll columns after encoding:")
print(list(df.columns))

Shape after one hot encoding: (7043, 31)

All columns after encoding:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [ ]:
## 4. Separate Features and Target Variable

In [10]:
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f"Features shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")
print(f"\nTarget distribution :")
print(y.value_counts())
print(f"Churn rate: {y.mean()*100:.1f}%")

Features shape (X): (7043, 30)
Target shape (y): (7043,)

Target distribution :
Churn
0    5174
1    1869
Name: count, dtype: int64
Churn rate: 26.5%


## 5. Train Test Split

We split BEFORE applying SMOTE to prevent data leakage.
SMOTE is only applied to training data — never test data.
The test set must represent real unseen customer data.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # ensures same churn ratio in both splits
)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set: {X_test.shape[0]:,} rows")
print(f"\nTraining churn distribution:")
print(y_train.value_counts())
print(f"\nTest churn distribution:")
print(y_test.value_counts())

Training set: 5,634 rows
Test set: 1,409 rows

Training churn distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64

Test churn distribution:
Churn
0    1035
1     374
Name: count, dtype: int64


## 6. Handle Class Imbalance — SMOTE

SMOTE (Synthetic Minority Oversampling Technique) generates
synthetic examples of churned customers in the training set
to balance the class distribution.

Applied to TRAINING DATA ONLY.

In [12]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(f"  Training samples: {X_train.shape[0]:,}")
print(f"  Churn (1): {y_train.sum():,} | "
      f"No Churn (0): {(y_train==0).sum():,}")

print("\nAfter SMOTE:")
print(f"  Training samples: {X_train_smote.shape[0]:,}")
churn_after = pd.Series(y_train_smote).sum()
no_churn_after = (pd.Series(y_train_smote)==0).sum()
print(f"  Churn (1): {churn_after:,} | "
      f"No Churn (0): {no_churn_after:,}")

Before SMOTE:
  Training samples: 5,634
  Churn (1): 1,495 | No Churn (0): 4,139

After SMOTE:
  Training samples: 8,278
  Churn (1): 4,139 | No Churn (0): 4,139


## 7. Feature Scaling

StandardScaler transforms numerical features to mean=0 
and standard deviation=1.

Fit on training data only — then transform both train and test.
This prevents data leakage from test set into training.

In [13]:
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

scaler = StandardScaler()

# Fit ONLY on training data
X_train_smote[numerical_cols] = scaler.fit_transform(
    X_train_smote[numerical_cols]
)

# Transform test data using training fit
X_test[numerical_cols] = scaler.transform(
    X_test[numerical_cols]
)

print("Scaling complete")
print("\nTraining set numerical stats after scaling:")
print(X_train_smote[numerical_cols].describe().round(3))

Scaling complete

Training set numerical stats after scaling:
         tenure  MonthlyCharges  TotalCharges
count  8278.000        8278.000      8278.000
mean      0.000           0.000        -0.000
std       1.000           1.000         1.000
min      -1.168          -1.734        -0.939
25%      -0.959          -0.789        -0.815
50%      -0.252           0.233        -0.414
75%       0.873           0.794         0.600
max       1.830           1.757         2.996


## 8. Save Processed Data

In [14]:
# Save the cleaned encoded dataframe before SMOTE
# This is used for EDA and SQL analytics
cleaned_df = pd.concat([X, y], axis=1)
cleaned_df.to_csv('../data/processed/telco_churn_cleaned.csv', 
                  index=False)

print("Cleaned dataset saved to data/processed/")
print(f"Shape: {cleaned_df.shape}")

Cleaned dataset saved to data/processed/
Shape: (7043, 31)
